# Actualización de Tablas Vida

## 1. Importación de paqueterías

In [1]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, text


## 2. Conexión a SQL Server

In [2]:

# -----------------------------
# 1️⃣ Conexión a SQL Server Azure
# -----------------------------
params_azure = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ikaldb.database.windows.net;"
    "DATABASE=actuarial;"
    "UID=CloudSA0052c2f7;"
    "PWD=ricostamales01#;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)
engine_azure = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_azure}",
    fast_executemany=True
)
print("✅ Conexión SQL Server AZURE establecida.")

# -----------------------------
# 2️⃣ Conexión a SQL Server Local
# -----------------------------
params_local = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=IKAL14\\SQLEXPRESS;"
    "DATABASE=actuarial;"
    "Trusted_Connection=yes;"
)
engine_local = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_local}",
    fast_executemany=True
)
print("✅ Conexión SQL Server Local establecida.")


✅ Conexión SQL Server AZURE establecida.
✅ Conexión SQL Server Local establecida.


## 3. Carga de datos

In [3]:
AñoMes = "202510"  # ← Cambia solo este valor cada mes
archivo_excel = rf"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{AñoMes[:4]}/{AñoMes}/{AñoMes}_Siniestros_Marine_PROCESADO.xlsx"
df_nuevo = pd.read_excel(archivo_excel)
print(f"✅ Datos leídos: '{archivo_excel}' ({len(df_nuevo):,} registros)")

✅ Datos leídos: 'C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202510/202510_Siniestros_Marine_PROCESADO.xlsx' (4,256 registros)


## 4. Limpieza y carga de tabla transporte

In [4]:

# -----------------------------
# 3️⃣ Actualizar tabla 'transporte'
# -----------------------------
print("📊 Actualizando tabla 'transporte' (mes actual)...")
df_nuevo.to_sql('transporte', con=engine_azure, if_exists='replace', index=False, schema='dbo')
df_nuevo.to_sql('transporte', con=engine_local, if_exists='replace', index=False, schema='dbo')
print(f"✅ Tabla 'transporte' reemplazada con {len(df_nuevo):,} registros")


📊 Actualizando tabla 'transporte' (mes actual)...
✅ Tabla 'transporte' reemplazada con 4,256 registros


## 5. Inserción y actualización de tabla transportehist

In [5]:

# =============================================================================
# 5️⃣ INSERCIÓN Y ACTUALIZACIÓN DE TABLA TRANSPORTEHIST CON MERGE
# =============================================================================

# 📋 PASO 1: Obtener estructura de tabla SQL existente
print("\n" + "="*80)
print("PASO 1: Obtener estructura de tabla SQL")
print("="*80)

query_columnas = """
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = 'transportehist' AND TABLE_SCHEMA = 'dbo'
ORDER BY ORDINAL_POSITION
"""
df_cols_azure = pd.read_sql(query_columnas, con=engine_azure)
df_cols_local = pd.read_sql(query_columnas, con=engine_local)

columnas_sql_list = df_cols_azure['COLUMN_NAME'].tolist()
columnas_sql_upper = {c.upper(): c for c in columnas_sql_list}
columnas_fecha_sql = set(df_cols_azure[df_cols_azure['DATA_TYPE'].isin(['date', 'datetime', 'datetime2'])]['COLUMN_NAME'].tolist())

def _get_len_map(df):
    m = (df['DATA_TYPE'].isin(['varchar', 'nvarchar', 'char', 'nchar']) &
         df['CHARACTER_MAXIMUM_LENGTH'].notna() & (df['CHARACTER_MAXIMUM_LENGTH'] > 0))
    return dict(zip(df.loc[m, 'COLUMN_NAME'], df.loc[m, 'CHARACTER_MAXIMUM_LENGTH'].astype(int)))

len_azure = _get_len_map(df_cols_azure)
len_local = _get_len_map(df_cols_local)
all_text_cols = set(len_azure) | set(len_local)
max_len_map = {col: min(v for v in [len_azure.get(col), len_local.get(col)] if v is not None)
               for col in all_text_cols}

columnas_clave = ['CLAIMS-ID', 'GROSS RESERVE']

print(f"✓ Columnas en tabla SQL (transportehist): {len(columnas_sql_list)}")
print(f"✓ Columnas fecha SQL: {sorted(columnas_fecha_sql)}")

rename_map = {col: columnas_sql_upper[col.upper()]
              for col in df_nuevo.columns
              if col.upper() in columnas_sql_upper and col != columnas_sql_upper[col.upper()]}
df_nuevo_renamed = df_nuevo.rename(columns=rename_map)

columnas_sql    = set(columnas_sql_list)
columnas_excel  = set(df_nuevo_renamed.columns.tolist())
columnas_validas = sorted(list(columnas_excel & columnas_sql))
print(f"✓ Columnas válidas para MERGE: {len(columnas_validas)}")

# 📋 PASO 2: Preparar datos del Excel
print("\n" + "="*80)
print("PASO 2: Preparar datos del Excel")
print("="*80)

faltantes_clave = [c for c in columnas_clave if c not in columnas_validas]
if faltantes_clave:
    raise ValueError(f"❌ Columnas PK faltantes: {faltantes_clave}")

df_transporte_limpio = df_nuevo_renamed[columnas_validas].copy()

# Convertir sólo las columnas que son fecha real (no enteros tipo año)
cols_fecha_presentes = [c for c in columnas_fecha_sql if c in df_transporte_limpio.columns]
for col in cols_fecha_presentes:
    ser = df_transporte_limpio[col]
    # Si ya es datetime, no tocar
    if pd.api.types.is_datetime64_any_dtype(ser):
        continue
    # Entero o float: puede ser serial Excel (>2100) o año (<=2100)
    if pd.api.types.is_integer_dtype(ser) or pd.api.types.is_float_dtype(ser):
        sample = ser.dropna()
        if len(sample) > 0 and sample.between(1900, 2100).all():
            # Año entero (2020, 2021…) → dejar como está; se corregirá en PASO 3b
            pass
        else:
            # Serial de Excel → convertir a fecha real
            df_transporte_limpio[col] = pd.to_datetime(ser, unit='D', origin='1899-12-30', errors='coerce')
    else:
        df_transporte_limpio[col] = pd.to_datetime(ser, errors='coerce')

for col, max_len in max_len_map.items():
    if col in df_transporte_limpio.columns and df_transporte_limpio[col].dtype == object:
        if df_transporte_limpio[col].str.len().gt(max_len).any():
            df_transporte_limpio[col] = df_transporte_limpio[col].str[:max_len]

print("✓ Datos preparados")

# 📋 PASO 3: Cargar datos a tabla temporal
print("\n" + "="*80)
print("PASO 3: Cargar datos a tabla temporal")
print("="*80)

df_transporte_limpio.to_sql('temp_transportehist_mes', con=engine_local, if_exists='replace', index=False, schema='dbo')
df_transporte_limpio.to_sql('temp_transportehist_mes', con=engine_azure, if_exists='replace', index=False, schema='dbo')
print(f"✓ Tabla temporal creada con {len(df_transporte_limpio):,} registros")

# 📋 PASO 3b: Corregir incompatibilidades de tipo entre temp y transportehist
print("\n" + "="*80)
print("PASO 3b: Verificar y corregir tipos")
print("="*80)

query_mismatch = """
SELECT t.COLUMN_NAME,
       t.DATA_TYPE AS tipo_hist,
       s.DATA_TYPE AS tipo_temp
FROM INFORMATION_SCHEMA.COLUMNS t
JOIN INFORMATION_SCHEMA.COLUMNS s
    ON t.COLUMN_NAME = s.COLUMN_NAME
   AND s.TABLE_NAME = 'temp_transportehist_mes'
   AND s.TABLE_SCHEMA = 'dbo'
WHERE t.TABLE_NAME = 'transportehist'
  AND t.TABLE_SCHEMA = 'dbo'
  AND t.DATA_TYPE <> s.DATA_TYPE
ORDER BY t.COLUMN_NAME
"""

mismatches = pd.read_sql(query_mismatch, con=engine_local)

# Columnas donde el histórico tiene DATE pero la temp tiene BIGINT → son años enteros
# Solución: migrar esas columnas a INT en transportehist
cols_bigint_to_date = mismatches[
    (mismatches['tipo_hist'] == 'date') &
    (mismatches['tipo_temp'] == 'bigint')
]['COLUMN_NAME'].tolist()

if cols_bigint_to_date:
    print(f"Migrando columnas de año DATE→INT en transportehist: {cols_bigint_to_date}")
    for col in cols_bigint_to_date:
        col_tmp = col.replace(" ", "_").replace("-", "_") + "_yr_tmp"
        for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
            with eng.begin() as conn:
                conn.execute(text(
                    f"IF COL_LENGTH('dbo.transportehist','{col_tmp}') IS NOT NULL "
                    f"ALTER TABLE dbo.transportehist DROP COLUMN [{col_tmp}]"
                ))
                conn.execute(text(f"ALTER TABLE dbo.transportehist ADD [{col_tmp}] INT NULL"))
                conn.execute(text(f"UPDATE dbo.transportehist SET [{col_tmp}] = YEAR([{col}])"))
                conn.execute(text(f"""
                    DECLARE @sql NVARCHAR(MAX) = ''
                    SELECT @sql = @sql + 'DROP INDEX [' + i.name + '] ON dbo.transportehist; '
                    FROM sys.indexes i
                    JOIN sys.index_columns ic
                        ON i.object_id = ic.object_id AND i.index_id = ic.index_id
                    JOIN sys.columns c
                        ON ic.object_id = c.object_id AND ic.column_id = c.column_id
                    WHERE OBJECT_NAME(i.object_id) = 'transportehist'
                      AND c.name = '{col}'
                      AND i.is_primary_key = 0
                      AND i.is_unique_constraint = 0
                    IF LEN(@sql) > 0 EXEC(@sql)
                """))
                conn.execute(text(f"ALTER TABLE dbo.transportehist DROP COLUMN [{col}]"))
                conn.execute(text(
                    f"EXEC sp_rename "
                    f"@objname=N'dbo.transportehist.{col_tmp}', "
                    f"@newname=N'{col}', @objtype=N'COLUMN'"
                ))
            print(f"  ✓ [{col}] → INT en {label}")

    # Verificar que ya no hay conflictos bigint/date
    remaining = pd.read_sql(query_mismatch, con=engine_local)
    still_bad = remaining[
        (remaining['tipo_hist'] == 'date') &
        (remaining['tipo_temp'] == 'bigint')
    ]
    if not still_bad.empty:
        print("❌ Siguen habiendo incompatibilidades bigint→date:")
        print(still_bad.to_string(index=False))
        raise ValueError("No se pudieron corregir todas las incompatibilidades. Ver tabla arriba.")

print("✓ Tipos corregidos, listo para MERGE")

# 📋 PASO 4: Construir y ejecutar MERGE
print("\n" + "="*80)
print("PASO 4: Ejecutar MERGE")
print("="*80)

on_clause   = ' AND '.join([f'target.[{c}] = source.[{c}]' for c in columnas_clave])
insert_cols = ', '.join([f'[{c}]' for c in columnas_validas])
insert_vals = ', '.join([f'source.[{c}]' for c in columnas_validas])
update_set  = ', '.join([f'target.[{c}] = source.[{c}]'
                         for c in columnas_validas if c not in columnas_clave])

merge_query = f"""
MERGE transportehist AS target
USING temp_transportehist_mes AS source
ON ({on_clause})
WHEN MATCHED THEN
    UPDATE SET {update_set}
WHEN NOT MATCHED THEN
    INSERT ({insert_cols}) VALUES ({insert_vals})
OUTPUT $action AS Action;
"""

with engine_local.begin() as conn:
    merge_output_local = conn.execute(text(merge_query)).fetchall()

with engine_azure.begin() as conn:
    merge_output_azure = conn.execute(text(merge_query)).fetchall()

updates = sum(1 for r in merge_output_azure if r[0] == 'UPDATE')
inserts = sum(1 for r in merge_output_azure if r[0] == 'INSERT')
print(f"✅ MERGE completado: {updates:,} actualizados, {inserts:,} insertados")

# 📋 PASO 5: Resumen y limpieza
print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)

reg_query = "SELECT COUNT(*) AS total FROM transportehist"
print(f" • Procesados del Excel: {len(df_transporte_limpio):,}")
print(f" • ACTUALIZADOS:  {updates:,}")
print(f" • INSERTADOS:    {inserts:,}")
print(f" • Total Azure:   {pd.read_sql(reg_query, con=engine_azure)['total'][0]:,}")
print(f" • Total Local:   {pd.read_sql(reg_query, con=engine_local)['total'][0]:,}")

for eng, label in [(engine_local, "Local"), (engine_azure, "Azure")]:
    with eng.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS temp_transportehist_mes"))
    print(f"🧹 Tabla temporal eliminada en {label}")

print("="*80)



PASO 1: Obtener estructura de tabla SQL
✓ Columnas en tabla SQL (transportehist): 38
✓ Columnas fecha SQL: ['DATE OF LOSS', 'DATE OF REPORT', 'MES CARGA', 'POLICY PERIOD END DATE', 'POLICY PERIOD START DATE']
✓ Columnas válidas para MERGE: 38

PASO 2: Preparar datos del Excel
✓ Datos preparados

PASO 3: Cargar datos a tabla temporal
✓ Tabla temporal creada con 4,256 registros

PASO 3b: Verificar y corregir tipos
✓ Tipos corregidos, listo para MERGE

PASO 4: Ejecutar MERGE


ProgrammingError: (pyodbc.ProgrammingError) ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The MERGE statement attempted to UPDATE or DELETE the same row more than once. This happens when a target row matches more than one source row. A MERGE statement cannot UPDATE/DELETE the same row of the target table multiple times. Refine the ON clause to ensure a target row matches at most one source row, or use the GROUP BY clause to group the source rows. (8672) (SQLFetch)')
(Background on this error at: https://sqlalche.me/e/20/f405)